# 🚀 Entrenamiento LSTM de Clientes en Google Colab

Este notebook entrena modelos LSTM para predecir comportamiento de clientes usando **GPU gratis de Colab**.

## ⏱️ Tiempo estimado: 2-4 horas
## 💰 Costo: $0 (Gratis)

---

## 📋 Antes de empezar:

1. **Activar GPU**: Runtime → Change runtime type → Hardware accelerator → GPU (T4)
2. **Subir archivos**:
   - `train_all_customers_temporal.py` (tu script de entrenamiento)
   - `online_retail_2.xlsx` (tu dataset)
3. **Ejecutar celdas** en orden (Shift + Enter)

---

## 1️⃣ Verificar GPU y configuración

In [ ]:
import tensorflow as tf

print("="*80)
print("CONFIGURACIÓN DE COLAB")
print("="*80)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

if len(tf.config.list_physical_devices('GPU')) > 0:
    print("\n✅ GPU ACTIVADA - El entrenamiento será RÁPIDO")
else:
    print("\n⚠️ GPU NO DETECTADA - Ve a Runtime → Change runtime type → GPU")

!free -h
!df -h | head -2

## 2️⃣ Instalar dependencias

In [ ]:
!pip install -q openpyxl seaborn
print("\n✅ Dependencias instaladas")

## 3️⃣ Subir archivos necesarios

In [ ]:
from google.colab import files

print("📤 Sube estos 2 archivos:")
print("   1. train_all_customers_temporal.py")
print("   2. online_retail_2.xlsx")
print("\nHaz click en 'Choose Files'...\n")

uploaded = files.upload()

print("\n✅ Archivos subidos:")
!ls -lh

## 4️⃣ Crear estructura de directorios

In [ ]:
!mkdir -p data/processed
!mkdir -p models/temporal/customer/short
!mkdir -p models/temporal/customer/medium
!mkdir -p models/temporal/customer/long

!mv online_retail_2.xlsx data/processed/

print("✅ Estructura creada")
!ls -R

## 5️⃣ Importar script de entrenamiento

In [ ]:
import sys
sys.path.append('.')

try:
    from train_all_customers_temporal import CustomerTemporalAnalyzer, TemporalConfig
    print("✅ Script importado correctamente")
except Exception as e:
    print(f"❌ ERROR: {e}")

## 6️⃣ Cargar y preprocesar datos

In [ ]:
import warnings
warnings.filterwarnings('ignore')

print("Inicializando...")
analyzer = CustomerTemporalAnalyzer(
    data_path='data/processed/online_retail_2.xlsx',
    output_dir='models/temporal/customer'
)

print("\n📊 Cargando datos (2-5 min)...\n")
analyzer.load_and_preprocess_data()

print(f"\n✅ Datos cargados correctamente")

## 7️⃣ Calcular métricas RFM y segmentar clientes

In [ ]:
print("📊 Calculando segmentación RFM...\n")
rfm = analyzer.calculate_rfm_metrics()

print("\n📊 Segmentos RFM creados:")
print(rfm['Segment_Label'].value_counts())

print("\n✅ Segmentación completada")

## 8️⃣ Generar secuencias temporales de clientes

In [ ]:
print("📈 Generando secuencias temporales (esto puede tomar 5-10 min)...\n")
customers = analyzer.generate_customer_sequences(min_transactions=5)

print(f"\n✅ Secuencias generadas para {len(customers):,} clientes")

## 9️⃣ Entrenar TODOS los horizontes (SHORT, MEDIUM, LONG)

**Esta celda entrenará los 3 horizontes automáticamente. Tomará 2-4 horas.**

In [ ]:
import time

print("🚀 ENTRENANDO TODOS LOS HORIZONTES...")
print("Esto tomará 2-4 horas en total.\n")
print("=" * 80)

start_total = time.time()

# Entrenar los 3 horizontes
results = analyzer.train_all_horizons()

duration = (time.time() - start_total) / 60

print("\n" + "=" * 80)
print(f"✅ ENTRENAMIENTO COMPLETADO en {duration:.1f} minutos")
print("=" * 80)

## 🔟 Revisar modelos entrenados

In [ ]:
print("📊 RESUMEN DE MODELOS ENTRENADOS\n")
print("=" * 80)

!ls -lh models/temporal/customer/short/
print()
!ls -lh models/temporal/customer/medium/
print()
!ls -lh models/temporal/customer/long/

print("\n✅ Todos los modelos entrenados y guardados!")

## 1️⃣1️⃣ Descargar modelos

**Descarga el ZIP con todos los modelos entrenados para usarlos localmente.**

In [ ]:
from google.colab import files

print("📦 Comprimiendo modelos...")
!cd models/temporal && zip -r customer_models.zip customer/

print("\n📥 Iniciando descarga...")
files.download('models/temporal/customer_models.zip')

print("\n✅ Descarga iniciada")
print("📁 Descomprime en: E:\\Codigos\\Proyecto Final\\models\\temporal\\")